# ACF, timescale analysis

## A. Analytical toy model, no noise

Trying to extract parameters from toy model
$$
\hat{C}_1(\tau) = 0.7\,\text{e}^{-\tau/1440} + 0.1\,\text{e}^{-\tau/60} + 0.1\,\text{e}^{-\tau/26} + 0.1\,\text{e}^{-\tau/2}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vac_metad.acf.toy import toy_model

toy = toy_model(frametime=1.0, num_frames=20_000)

# ts = acf_timescales(acf)
# sm = slow_mode_analysis(acf, split_window=(300.0, 1200.0))
# print(ts)
# print(sm)
# for n in (2, 3, 4):
#     m = fit_multi_exponential(acf, n, sum_to_one=True)
#     print(n, m.coeffs.round(3), m.timescales.round(1), f"cond={m.cond:.2g}")

In [ ]:
from vac_metad.dataclass import DataClass

DEFAULT_COEFFS = (0.7, 0.1, 0.1, 0.1)
DEFAULT_TIMESCALES = (1850, 60, 26, 2)


def acf_terms(t, coeffs, timescales):
    """Return the individual ACF terms, a_i * exp(-t / tau_i), at time(s) t.

    t may be a scalar or a 1D array, in ps. The first axis of the result
    indexes the terms:
        scalar t -> shape (n_terms,)
        1D t     -> shape (n_terms, len(t))
    """
    return np.array([a * np.exp(-t / ti) for a, ti in zip(coeffs, timescales)])


def toy_model(
    frametime,
    num_lags,
    num_frames=None,
    coeffs=DEFAULT_COEFFS,
    timescales=DEFAULT_TIMESCALES,
):
    """Exact toy ACF, C(t) = sum_i a_i exp(-t / t_i)

    Parameters
    ----------
    frametime : time separating frames (ps, e.g.)
    num_lags : number of frames for ACF
    num_frames : length of a pretend trajectory, used only for noise
        estimates.  None means noise-free.
    coeffs : a_i terms
    timescales : t_1 terms
    """
    t = frametime * np.arange(num_lags)
    terms = acf_terms(t, coeffs, timescales)
    return DataClass(
        t=t,
        ct=terms.sum(axis=0),
        frametime=frametime,
        num_frames=num_frames,
        coeffs=np.array(coeffs),
        timescales=np.array(timescales),
        terms=terms,
    )